# Task 5 — Select 10 CUAD Categories
- Review support for all 41 CUAD categories.
- Count positive training contracts and context groups for each category.
- Compare category support with Sebastian’s priority themes:
  - Financial implications
  - Renewal terms
  - Metadata
  - Liability
- Create a shortlist of candidate categories.
- Select the final 10 categories with sufficient training support and business relevance.
- Document any exceptions or lower-support categories that are still selected.

In [3]:
# Import the libraries needed to load CSV and Parquet files directly from GitHub
from utils import load_file_from_github

In [4]:
# Load the Task 3 category-support results
category_support = load_file_from_github(
    "notebooks/task3_outputs/category_support.csv",
    branch="gate-1-verification-reproducibility"
)

# Load the normalized annotation-set table created in Task 1
annotation_sets = load_file_from_github(
    "notebooks/split_data/annotation_sets.parquet",
    branch="gate-1-verification-reproducibility"
)

# Load the normalized documents table created in Task 1
# This table provides the context_group_id used for the split
documents = load_file_from_github(
    "notebooks/split_data/documents.parquet",
    branch="gate-1-verification-reproducibility"
)

# Load the frozen train/validation split created in Task 2
split_map = load_file_from_github(
    "notebooks/frozen-split_data/frozen-split.parquet",
    branch="gate-1-verification-reproducibility"
)


# Verify that every required Task 5 input loaded correctly
print("category_support:", category_support.shape)
print("annotation_sets:", annotation_sets.shape)
print("documents:", documents.shape)
print("split_map:", split_map.shape)

category_support: (41, 3)
annotation_sets: (16728, 4)
documents: (408, 4)
split_map: (407, 2)


In [5]:
print(category_support.columns.tolist())
display(category_support.head())

['category_id', 'num_supporting_contracts', 'category_name']


,category_id,num_supporting_contracts,category_name
0,document_name,325,Document Name
1,parties,324,Parties
2,agreement_date,300,Agreement Date
3,governing_law,274,Governing Law
4,expiration_date,263,Expiration Date


### Step 1 — Review Training-Data Support

Review all 41 CUAD categories and compare how many positive training contracts support each category. This helps identify categories with enough data to be considered for the final selection.

In [6]:
# Make a copy of the Task 3 category-support table so we do not modify the original
task5_support = category_support.copy()

# Total number of contracts/documents in the training set
TOTAL_TRAIN_CONTRACTS = 325

# Calculate the percentage of training contracts that contain each category
task5_support["pct_training_contracts"] = (
    task5_support["num_supporting_contracts"]
    / TOTAL_TRAIN_CONTRACTS
    * 100
).round(2)

# Sort categories from highest to lowest training support
task5_support = task5_support.sort_values(
    by="num_supporting_contracts",
    ascending=False
).reset_index(drop=True)

# Display all 41 categories and their training support
display(task5_support)

,category_id,num_supporting_contracts,category_name,pct_training_contracts
0,document_name,325,Document Name,100.00
1,parties,324,Parties,99.69
2,agreement_date,300,Agreement Date,92.31
3,governing_law,274,Governing Law,84.31
4,expiration_date,263,Expiration Date,80.92
5,effective_date,253,Effective Date,77.85
6,anti_assignment,235,Anti-Assignment,72.31
7,cap_on_liability,184,Cap On Liability,56.62
8,license_grant,163,License Grant,50.15
9,audit_rights,139,Audit Rights,42.77


### Step 2 — Count Positive Context Groups

Count the number of unique training `context_group_id`s that contain each CUAD category. This confirms that category support is spread across independent training groups.

In [7]:
# Create a table that connects each annotation set to its document's context group
# This lets us count how many unique training context groups support each category
category_context_data = annotation_sets.merge(
    documents[["contract_id", "context_group_id"]],
    on="contract_id",
    how="left"
)

# Add the frozen train/validation split label to each row
category_context_data = category_context_data.merge(
    split_map[["context_group_id", "split"]],
    on="context_group_id",
    how="left"
)

# Keep only rows from the training split
train_category_context = category_context_data[
    category_context_data["split"] == "train"
].copy()

# Keep only positive annotations:
# is_impossible == False means the category is actually present in the contract
positive_train_context = train_category_context[
    train_category_context["is_impossible"] == False
].copy()

# Count the number of unique positive context groups for each category
context_group_support = (
    positive_train_context
    .groupby("category_id")["context_group_id"]
    .nunique()
    .reset_index(name="num_supporting_context_groups")
)

# Add the context-group counts to our existing Task 5 support table
task5_support = task5_support.merge(
    context_group_support,
    on="category_id",
    how="left"
)

# Categories with no positive context groups would appear as NaN,
# so replace missing values with 0
task5_support["num_supporting_context_groups"] = (
    task5_support["num_supporting_context_groups"]
    .fillna(0)
    .astype(int)
)

# Display the updated support table
display(task5_support)

,category_id,num_supporting_contracts,category_name,pct_training_contracts,num_supporting_context_groups
0,document_name,325,Document Name,100.00,324
1,parties,324,Parties,99.69,323
2,agreement_date,300,Agreement Date,92.31,299
3,governing_law,274,Governing Law,84.31,273
4,expiration_date,263,Expiration Date,80.92,262
5,effective_date,253,Effective Date,77.85,252
6,anti_assignment,235,Anti-Assignment,72.31,235
7,cap_on_liability,184,Cap On Liability,56.62,184
8,license_grant,163,License Grant,50.15,163
9,audit_rights,139,Audit Rights,42.77,139


### Step 3 — Examine the Support Distribution

Review the highest- and lowest-supported categories to understand how training support is distributed before setting a threshold for candidate selection.

In [8]:
# Display basic statistics for the number of supporting training contracts
# and supporting context groups across all 41 CUAD categories
support_summary = task5_support[
    ["num_supporting_contracts", "num_supporting_context_groups"]
].describe()

display(support_summary)


# Display the 10 categories with the LOWEST training support
# This helps identify categories that may not have enough examples for modeling
lowest_support = task5_support.sort_values(
    by="num_supporting_context_groups",
    ascending=True
).head(10)

print("10 Lowest-Supported Categories:")
display(
    lowest_support[
        [
            "category_id",
            "category_name",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts"
        ]
    ]
)


# Display the 10 categories with the HIGHEST training support
# This gives us a comparison point for the strongest-supported categories
highest_support = task5_support.sort_values(
    by="num_supporting_context_groups",
    ascending=False
).head(10)

print("10 Highest-Supported Categories:")
display(
    highest_support[
        [
            "category_id",
            "category_name",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts"
        ]
    ]
)

,num_supporting_contracts,num_supporting_context_groups
count,41.000000,41.000000
mean,105.292683,105.073171
std,92.332346,92.052808
min,10.000000,10.000000
25%,38.000000,38.000000
50%,77.000000,77.000000
75%,123.000000,122.000000
max,325.000000,324.000000


10 Lowest-Supported Categories:


,category_id,category_name,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts
40,source_code_escrow,Source Code Escrow,10,10,3.08
39,unlimited_all_you_can_eat_license,Unlimited/All-You-Can-Eat-License,11,11,3.38
38,price_restrictions,Price Restrictions,12,12,3.69
37,affiliate_license_licensor,Affiliate License-Licensor,14,14,4.31
36,most_favored_nation,Most Favored Nation,20,20,6.15
35,third_party_beneficiary,Third Party Beneficiary,21,21,6.46
34,no_solicit_of_customers,No-Solicit Of Customers,22,22,6.77
33,non_disparagement,Non-Disparagement,25,25,7.69
32,joint_ip_ownership,Joint Ip Ownership,31,31,9.54
29,no_solicit_of_employees,No-Solicit Of Employees,39,38,12.00


10 Highest-Supported Categories:


,category_id,category_name,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts
0,document_name,Document Name,325,324,100.00
1,parties,Parties,324,323,99.69
2,agreement_date,Agreement Date,300,299,92.31
3,governing_law,Governing Law,274,273,84.31
4,expiration_date,Expiration Date,263,262,80.92
5,effective_date,Effective Date,253,252,77.85
6,anti_assignment,Anti-Assignment,235,235,72.31
7,cap_on_liability,Cap On Liability,184,184,56.62
8,license_grant,License Grant,163,163,50.15
9,audit_rights,Audit Rights,139,139,42.77


### Step 4 — Classify Training Support

Use the observed category-support distribution to group categories into strong, moderate, and low-support tiers. These tiers are only sed as a review guide, not as a fixed modeling requirement.

In [9]:
# Classify each CUAD category based on the training support distribution.
# The thresholds come from the observed quartiles:
# - Median = 77 supporting context groups
# - 25th percentile = 38 supporting context groups

def classify_support(num_context_groups):
    # Categories at or above the median are considered strongly supported
    if num_context_groups >= 77:
        return "Strong"

    # Categories between the 25th percentile and median have moderate support
    elif num_context_groups >= 38:
        return "Moderate"

    # Categories below the 25th percentile are flagged as low support
    else:
        return "Low"


# Apply the support classification to all 41 categories
task5_support["support_level"] = (
    task5_support["num_supporting_context_groups"]
    .apply(classify_support)
)

# Display the full category-support table with the new support level
display(
    task5_support[
        [
            "category_id",
            "category_name",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level"
        ]
    ]
)

# Show how many categories fall into each support tier
print("Number of categories in each support level:")
print(task5_support["support_level"].value_counts())

,category_id,category_name,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level
0,document_name,Document Name,325,324,100.00,Strong
1,parties,Parties,324,323,99.69,Strong
2,agreement_date,Agreement Date,300,299,92.31,Strong
3,governing_law,Governing Law,274,273,84.31,Strong
4,expiration_date,Expiration Date,263,262,80.92,Strong
5,effective_date,Effective Date,253,252,77.85,Strong
6,anti_assignment,Anti-Assignment,235,235,72.31,Strong
7,cap_on_liability,Cap On Liability,184,184,56.62,Strong
8,license_grant,License Grant,163,163,50.15,Strong
9,audit_rights,Audit Rights,139,139,42.77,Strong


Number of categories in each support level:
support_level
Strong      21
Moderate    11
Low          9
Name: count, dtype: int64


### Step 5 — Map Categories to Broader Business-Priority Themes

Map CUAD categories to broader contract-review priorities. Sebastian's four suggested themes are: Financial implications, Renewal terms, Metadata, Liability. dditional themes are included based on the project's goal of identifying important contractual risks and obligations.

In [10]:
# Create a broader business-priority map.
# Sebastian's four suggested themes are included:
# Financial, Renewal, Metadata, and Liability.
#
# Additional themes are added because they are also relevant to
# contract review, legal risk, and business decision-making.

theme_map = {

    # ---------------------------------------------------------
    # Metadata
    # Basic information used to identify and understand a contract.
    # ---------------------------------------------------------
    "document_name": "Metadata",
    "parties": "Metadata",
    "agreement_date": "Metadata",
    "effective_date": "Metadata",
    "expiration_date": "Metadata",
    "governing_law": "Metadata",

    # ---------------------------------------------------------
    # Renewal
    # Clauses that determine whether and how a contract renews.
    # ---------------------------------------------------------
    "renewal_term": "Renewal",
    "notice_period_to_terminate_renewal": "Renewal",

    # ---------------------------------------------------------
    # Financial
    # Clauses that can create direct financial obligations,
    # restrictions, commitments, or monetary exposure.
    # ---------------------------------------------------------
    "revenue_profit_sharing": "Financial",
    "minimum_commitment": "Financial",
    "volume_restriction": "Financial",
    "price_restrictions": "Financial",
    "liquidated_damages": "Financial",

    # ---------------------------------------------------------
    # Liability
    # Clauses that determine responsibility for losses,
    # damages, insurance, or legal exposure.
    # ---------------------------------------------------------
    "cap_on_liability": "Liability",
    "uncapped_liability": "Liability",
    "insurance": "Liability",
    "warranty_duration": "Liability",
    "covenant_not_to_sue": "Liability",

    # ---------------------------------------------------------
    # Termination / Exit
    # Clauses describing how a contract can end and what
    # obligations continue after termination.
    # ---------------------------------------------------------
    "termination_for_convenience": "Termination / Exit",
    "post_termination_services": "Termination / Exit",

    # ---------------------------------------------------------
    # IP / Licensing
    # Clauses involving intellectual-property ownership,
    # licensing rights, and use of technology or content.
    # ---------------------------------------------------------
    "license_grant": "IP / Licensing",
    "ip_ownership_assignment": "IP / Licensing",
    "joint_ip_ownership": "IP / Licensing",
    "non_transferable_license": "IP / Licensing",
    "irrevocable_or_perpetual_license": "IP / Licensing",
    "source_code_escrow": "IP / Licensing",

    # ---------------------------------------------------------
    # Competitive Restrictions
    # Clauses that limit competition or business relationships.
    # ---------------------------------------------------------
    "exclusivity": "Competitive Restrictions",
    "non_compete": "Competitive Restrictions",
    "competitive_restriction_exception": "Competitive Restrictions",
    "no_solicit_of_employees": "Competitive Restrictions",
    "no_solicit_of_customers": "Competitive Restrictions",
    "non_disparagement": "Competitive Restrictions",

    # ---------------------------------------------------------
    # Transfer / Control
    # Clauses affecting assignment, ownership changes,
    # or transfer of contractual rights.
    # ---------------------------------------------------------
    "anti_assignment": "Transfer / Control",
    "change_of_control": "Transfer / Control",

    # ---------------------------------------------------------
    # Audit / Compliance
    # Clauses that allow monitoring, verification, or review
    # of contractual obligations.
    # ---------------------------------------------------------
    "audit_rights": "Audit / Compliance"
}


# Map each CUAD category to one of the business-priority themes.
# Categories that are not currently assigned to a priority theme
# are temporarily labeled "Other".
task5_support["business_theme"] = (
    task5_support["category_id"]
    .map(theme_map)
    .fillna("Other")
)


# Display all 41 categories so we can review the theme assignments
# alongside their training-data support.
display(
    task5_support[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level"
        ]
    ]
)


# Show how many CUAD categories currently fall into each theme.
print("Number of categories by business theme:")
print(task5_support["business_theme"].value_counts())


# Show any categories that have not yet been assigned to one of
# our proposed business-priority themes.
# We will review these before creating the candidate shortlist.
print("\nCategories currently labeled as Other:")

display(
    task5_support[
        task5_support["business_theme"] == "Other"
    ][
        [
            "category_id",
            "category_name",
            "num_supporting_context_groups",
            "support_level"
        ]
    ]
)

,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level
0,document_name,Document Name,Metadata,325,324,100.00,Strong
1,parties,Parties,Metadata,324,323,99.69,Strong
2,agreement_date,Agreement Date,Metadata,300,299,92.31,Strong
3,governing_law,Governing Law,Metadata,274,273,84.31,Strong
4,expiration_date,Expiration Date,Metadata,263,262,80.92,Strong
5,effective_date,Effective Date,Metadata,253,252,77.85,Strong
6,anti_assignment,Anti-Assignment,Transfer / Control,235,235,72.31,Strong
7,cap_on_liability,Cap On Liability,Liability,184,184,56.62,Strong
8,license_grant,License Grant,IP / Licensing,163,163,50.15,Strong
9,audit_rights,Audit Rights,Audit / Compliance,139,139,42.77,Strong


Number of categories by business theme:
business_theme
Metadata                    6
IP / Licensing              6
Competitive Restrictions    6
Other                       6
Liability                   5
Financial                   5
Transfer / Control          2
Termination / Exit          2
Renewal                     2
Audit / Compliance          1
Name: count, dtype: int64

Categories currently labeled as Other:


,category_id,category_name,num_supporting_context_groups,support_level
24,rofr_rofo_rofn,Rofr/Rofo/Rofn,54,Moderate
30,affiliate_license_licensee,Affiliate License-Licensee,38,Moderate
35,third_party_beneficiary,Third Party Beneficiary,21,Low
36,most_favored_nation,Most Favored Nation,20,Low
37,affiliate_license_licensor,Affiliate License-Licensor,14,Low
39,unlimited_all_you_can_eat_license,Unlimited/All-You-Can-Eat-License,11,Low


In [11]:
# ---------------------------------------------------------
# Refine categories that were previously labeled "Other"
# so that every CUAD category has been intentionally reviewed.
# ---------------------------------------------------------

additional_theme_map = {

    # Rights involving first refusal, first offer, or first negotiation
    "rofr_rofo_rofn": "Strategic / Commercial Rights",

    # Additional licensing-related categories
    "affiliate_license_licensee": "IP / Licensing",
    "affiliate_license_licensor": "IP / Licensing",
    "unlimited_all_you_can_eat_license": "IP / Licensing",

    # Commercial terms that can affect pricing or negotiated benefits
    "most_favored_nation": "Financial / Commercial",

    # Rights granted to parties who are not direct signatories
    "third_party_beneficiary": "Third-Party Rights"
}


# Update the business theme only for categories included
# in the additional mapping above.
task5_support["business_theme"] = task5_support.apply(
    lambda row: additional_theme_map.get(
        row["category_id"],
        row["business_theme"]
    ),
    axis=1
)


# Check whether any categories are still labeled as "Other".
remaining_other = task5_support[
    task5_support["business_theme"] == "Other"
]

print("Categories still labeled as Other:", len(remaining_other))
display(remaining_other)

Categories still labeled as Other: 0


,category_id,num_supporting_contracts,category_name,pct_training_contracts,num_supporting_context_groups,support_level,business_theme


### Step 6 — Propose Initial Candidate Categories

Create an initial candidate pool from the 41 CUAD categories using training-data support and business relevance.

- Retain categories with Strong or Moderate support.
- Temporarily exclude Low-support categories from the main candidate pool.
- Keep Low-support categories available for later review as possible exceptions.
- Compare candidates across the different business-priority themes before selecting the final 10.

In [12]:
# ---------------------------------------------------------
# Create the initial candidate pool.
#
# Categories classified as Strong or Moderate are retained
# because they have enough positive training examples to be
# considered further.
#
# Low-support categories are not deleted. They are separated
# so they can still be reviewed later as possible exceptions.
# ---------------------------------------------------------

candidate_categories = task5_support[
    task5_support["support_level"].isin(["Strong", "Moderate"])
].copy()


# Sort the candidate categories by:
# 1. Business theme
# 2. Number of supporting context groups
#
# This makes it easier to compare categories within each theme.
candidate_categories = candidate_categories.sort_values(
    by=["business_theme", "num_supporting_context_groups"],
    ascending=[True, False]
).reset_index(drop=True)


# Display the initial candidate pool.
display(
    candidate_categories[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level"
        ]
    ]
)


# Print the total number of categories that remain
# after excluding Low-support categories.
print(
    "Number of initial candidate categories:",
    len(candidate_categories)
)


# Show how many candidates come from each business theme.
# This helps us see whether some themes are over- or under-represented.
print("\nCandidate categories by business theme:")

display(
    candidate_categories["business_theme"]
    .value_counts()
    .rename_axis("business_theme")
    .reset_index(name="num_candidates")
)

,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level
0,audit_rights,Audit Rights,Audit / Compliance,139,139,42.77,Strong
1,exclusivity,Exclusivity,Competitive Restrictions,117,117,36.00,Strong
2,non_compete,Non-Compete,Competitive Restrictions,77,77,23.69,Strong
3,competitive_restriction_exception,Competitive Restriction Exception,Competitive Restrictions,48,48,14.77,Moderate
4,no_solicit_of_employees,No-Solicit Of Employees,Competitive Restrictions,39,38,12.00,Moderate
5,revenue_profit_sharing,Revenue/Profit Sharing,Financial,103,103,31.69,Strong
6,minimum_commitment,Minimum Commitment,Financial,101,101,31.08,Strong
7,volume_restriction,Volume Restriction,Financial,52,52,16.00,Moderate
8,liquidated_damages,Liquidated Damages,Financial,38,38,11.69,Moderate
9,license_grant,License Grant,IP / Licensing,163,163,50.15,Strong


Number of initial candidate categories: 32

Candidate categories by business theme:


,business_theme,num_candidates
0,Metadata,6
1,IP / Licensing,5
2,Liability,5
3,Competitive Restrictions,4
4,Financial,4
5,Renewal,2
6,Termination / Exit,2
7,Transfer / Control,2
8,Audit / Compliance,1
9,Strategic / Commercial Rights,1


In [13]:
# ---------------------------------------------------------
# Separate the Low-support categories.
#
# These categories are not currently part of the main candidate
# pool, but they remain available for review if there is a strong
# business reason to include one in the final 10.
# ---------------------------------------------------------

low_support_exceptions = task5_support[
    task5_support["support_level"] == "Low"
].copy()


# Sort Low-support categories from highest to lowest support
# so the strongest potential exceptions appear first.
low_support_exceptions = low_support_exceptions.sort_values(
    by="num_supporting_context_groups",
    ascending=False
)


# Display possible exception categories.
print("Low-support categories to review as possible exceptions:")

display(
    low_support_exceptions[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level"
        ]
    ]
)

Low-support categories to review as possible exceptions:


,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level
32,joint_ip_ownership,Joint Ip Ownership,IP / Licensing,31,31,9.54,Low
33,non_disparagement,Non-Disparagement,Competitive Restrictions,25,25,7.69,Low
34,no_solicit_of_customers,No-Solicit Of Customers,Competitive Restrictions,22,22,6.77,Low
35,third_party_beneficiary,Third Party Beneficiary,Third-Party Rights,21,21,6.46,Low
36,most_favored_nation,Most Favored Nation,Financial / Commercial,20,20,6.15,Low
37,affiliate_license_licensor,Affiliate License-Licensor,IP / Licensing,14,14,4.31,Low
38,price_restrictions,Price Restrictions,Financial,12,12,3.69,Low
39,unlimited_all_you_can_eat_license,Unlimited/All-You-Can-Eat-License,IP / Licensing,11,11,3.38,Low
40,source_code_escrow,Source Code Escrow,IP / Licensing,10,10,3.08,Low


### Step 7 — Rank Candidates Within Each Business Theme

Rank the candidate categories within their business themes using positive training context-group support. This helps compare categories fairly across different contract-review priorities before selecting the final 10.

In [14]:
# ---------------------------------------------------------
# Rank candidate categories within each business theme.
#
# A rank of 1 means the category has the highest number of
# supporting training context groups within that theme.
# ---------------------------------------------------------

candidate_categories["theme_support_rank"] = (
    candidate_categories
    .groupby("business_theme")["num_supporting_context_groups"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


# Sort the table so that each business theme is grouped together
# and its strongest-supported categories appear first.
ranked_candidates = candidate_categories.sort_values(
    by=[
        "business_theme",
        "theme_support_rank",
        "num_supporting_context_groups"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)


# Display the ranked candidate pool.
display(
    ranked_candidates[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level",
            "theme_support_rank"
        ]
    ]
)

,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level,theme_support_rank
0,audit_rights,Audit Rights,Audit / Compliance,139,139,42.77,Strong,1
1,exclusivity,Exclusivity,Competitive Restrictions,117,117,36.00,Strong,1
2,non_compete,Non-Compete,Competitive Restrictions,77,77,23.69,Strong,2
3,competitive_restriction_exception,Competitive Restriction Exception,Competitive Restrictions,48,48,14.77,Moderate,3
4,no_solicit_of_employees,No-Solicit Of Employees,Competitive Restrictions,39,38,12.00,Moderate,4
5,revenue_profit_sharing,Revenue/Profit Sharing,Financial,103,103,31.69,Strong,1
6,minimum_commitment,Minimum Commitment,Financial,101,101,31.08,Strong,2
7,volume_restriction,Volume Restriction,Financial,52,52,16.00,Moderate,3
8,liquidated_damages,Liquidated Damages,Financial,38,38,11.69,Moderate,4
9,license_grant,License Grant,IP / Licensing,163,163,50.15,Strong,1


In [15]:
# ---------------------------------------------------------
# Create a smaller comparison shortlist.
#
# Keep the two strongest-supported categories from each theme.
# This reduces the 32-category candidate pool while preserving
# representation across the project's different business priorities.
# ---------------------------------------------------------

comparison_shortlist = ranked_candidates[
    ranked_candidates["theme_support_rank"] <= 2
].copy()


# Sort the shortlist by training support so the strongest
# candidates are easy to identify.
comparison_shortlist = comparison_shortlist.sort_values(
    by="num_supporting_context_groups",
    ascending=False
).reset_index(drop=True)


# Display the smaller candidate shortlist.
display(
    comparison_shortlist[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level",
            "theme_support_rank"
        ]
    ]
)

print(
    "Number of categories in comparison shortlist:",
    len(comparison_shortlist)
)

,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level,theme_support_rank
0,document_name,Document Name,Metadata,325,324,100.00,Strong,1
1,parties,Parties,Metadata,324,323,99.69,Strong,2
2,anti_assignment,Anti-Assignment,Transfer / Control,235,235,72.31,Strong,1
3,cap_on_liability,Cap On Liability,Liability,184,184,56.62,Strong,1
4,license_grant,License Grant,IP / Licensing,163,163,50.15,Strong,1
5,audit_rights,Audit Rights,Audit / Compliance,139,139,42.77,Strong,1
6,termination_for_convenience,Termination For Convenience,Termination / Exit,123,122,37.85,Strong,1
7,post_termination_services,Post-Termination Services,Termination / Exit,122,122,37.54,Strong,1
8,renewal_term,Renewal Term,Renewal,120,120,36.92,Strong,1
9,exclusivity,Exclusivity,Competitive Restrictions,117,117,36.00,Strong,1


Number of categories in comparison shortlist: 18


### Step 8 — Propose the Final 10 Categories

Select 10 categories from the candidate pool by balancing training-data support, business relevance, risk importance, and coverage across multiple contract-review themes. The final selection should not be based on frequency alone.

In [16]:
# ---------------------------------------------------------
# Proposed final 10 categories.
#
# These categories were selected by balancing:
# - Training-data support
# - Business relevance
# - Contract-review risk
# - Representation across different business themes
# ---------------------------------------------------------

proposed_final_10 = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity"
]


# Pull the proposed categories from the full support table
# so that we retain all training-support information.
final_10_table = task5_support[
    task5_support["category_id"].isin(proposed_final_10)
].copy()


# Sort the proposed categories by business theme
# to make the final table easier to review.
final_10_table = final_10_table.sort_values(
    by="business_theme"
).reset_index(drop=True)


# Display the proposed final selection.
display(
    final_10_table[
        [
            "category_id",
            "category_name",
            "business_theme",
            "num_supporting_contracts",
            "num_supporting_context_groups",
            "pct_training_contracts",
            "support_level"
        ]
    ]
)

# Confirm exactly 10 categories were selected.
print("Number of proposed categories:", len(final_10_table))

,category_id,category_name,business_theme,num_supporting_contracts,num_supporting_context_groups,pct_training_contracts,support_level
0,audit_rights,Audit Rights,Audit / Compliance,139,139,42.77,Strong
1,exclusivity,Exclusivity,Competitive Restrictions,117,117,36.00,Strong
2,revenue_profit_sharing,Revenue/Profit Sharing,Financial,103,103,31.69,Strong
3,license_grant,License Grant,IP / Licensing,163,163,50.15,Strong
4,cap_on_liability,Cap On Liability,Liability,184,184,56.62,Strong
5,uncapped_liability,Uncapped Liability,Liability,78,78,24.00,Strong
6,governing_law,Governing Law,Metadata,274,273,84.31,Strong
7,renewal_term,Renewal Term,Renewal,120,120,36.92,Strong
8,termination_for_convenience,Termination For Convenience,Termination / Exit,123,122,37.85,Strong
9,anti_assignment,Anti-Assignment,Transfer / Control,235,235,72.31,Strong


Number of proposed categories: 10


### Step 9 — Final Selection Justification

The proposed 10 categories were selected based on strong training-data support, business relevance, and coverage across important contract-review risks.

| Category | Theme | Short Justification |
|---|---|---|
| Governing Law | Metadata | Identifies the legal jurisdiction governing the contract. |
| Renewal Term | Renewal | Important for understanding ongoing contract obligations. |
| Revenue/Profit Sharing | Financial | Captures direct financial obligations between parties. |
| Cap on Liability | Liability | Defines limits on financial/legal exposure. |
| Uncapped Liability | Liability | Identifies potentially high-risk unlimited exposure. |
| Termination for Convenience | Termination / Exit | Shows whether a party can exit the contract easily. |
| Anti-Assignment | Transfer / Control | Controls whether contractual rights can be transferred. |
| Audit Rights | Audit / Compliance | Supports monitoring and verification of obligations. |
| License Grant | IP / Licensing | Defines important intellectual-property usage rights. |
| Exclusivity | Competitive Restrictions | Identifies restrictions on working with competitors or others. |

**Exceptions:** No low-support categories were included. All selected categories have Strong training support.